# 01 · The two test sets

What the test sets contain, and whether they can answer the questions. No model is used here.

- **MultiEURLEX** (government): EU laws in 23 languages, labelled with EuroVoc by the EU Publications Office.
- **HumSet** (humanitarian): excerpts from crisis documents in English, French and Spanish, labelled by analysts.

**Pass marks, written before running.**
- The EuroVoc tree reproduces the dataset's level 1 labels on at least 95% of test laws.
- No HumSet test document appears in the training split.
- If fewer than 50 test laws reach 20,000 words, the rollup test cannot speak to report-length documents, and this
  notebook raises.


In [1]:
from collections import Counter

import numpy as np
import pandas as pd

from tiered_taxonomy import datasets as ds
from tiered_taxonomy import experiment as ex
from tiered_taxonomy.config import data_root

root = data_root()
eurlex = ex.load_eurlex(root)
humset = ex.load_humset(root)
results = {}

## Sizes

In [2]:
rows = []
for data in (eurlex, humset):
    tree = data.taxonomy
    rows.append(
        {
            "test set": data.name,
            "train": len(data.train),
            "dev": len(data.dev),
            "test": len(data.test),
            "nodes": len(tree.nodes),
            "tier sizes": dict(sorted(Counter(tree.tiers.values()).items())),
            "nodes with 2+ parents": sum(len(node.parents) > 1 for node in tree.nodes.values()),
            "mean gold labels": round(float(np.mean([len(item.labels) for item in data.test])), 2),
        }
    )
results["sizes"] = rows
pd.DataFrame(rows)

,test set,train,dev,test,nodes,tier sizes,nodes with 2+ parents,mean gold labels
0,eurlex,55000,5000,5000,7372,"{1: 21, 2: 127, 3: 545, 4: 4013, 5: 2173, 6: 4...",244,6.22
1,humset,117435,16039,15147,75,"{1: 24, 2: 51}",0,2.34


## How long are the documents?

The UN evaluation reports this work is ultimately for have a median of about 33,000 words in the sample read earlier.

In [3]:
lengths = np.array([ds.word_count(item.text) for item in eurlex.test])
summary = {
    "median_words": int(np.median(lengths)),
    "at_least_5000": int((lengths >= 5000).sum()),
    "at_least_20000": int((lengths >= 20000).sum()),
    "test_documents": len(lengths),
}
if summary["at_least_20000"] < 50:
    raise ValueError("too few long laws for the rollup test to say anything about report-length documents")
results["eurlex_lengths"] = summary
summary

{'median_words': 509,
 'at_least_5000': 370,
 'at_least_20000': 63,
 'test_documents': 5000}

## Does today's EuroVoc tree match the dataset?

The dataset has labels at levels 1 to 3 but no tree. The tree comes from the EU Publications Office, and is newer than
the dataset. So every method's labels and all gold labels are rolled up through this same tree, and these figures are a
known limit on how far the tiers match what the dataset's authors had.

In [4]:
frame = ds.load_eurlex_rows(root, "test")
agreement = {}
for tier, column in ds.EURLEX_LEVELS.items():
    same = [
        eurlex.taxonomy.roll_up([label for label in labels if label in eurlex.taxonomy.nodes], tier)
        == set(level)
        for labels, level in zip(frame["all_levels"], frame[column], strict=True)
    ]
    agreement[f"level_{tier}"] = round(float(np.mean(same)), 3)
labels_in_data = {label for labels in frame["all_levels"] for label in labels}
agreement["test_labels_missing_from_tree"] = len(labels_in_data - set(eurlex.taxonomy.nodes))
if agreement["level_1"] < 0.95:
    raise ValueError("the tree does not match the dataset's top level")
results["eurovoc_agreement"] = agreement
agreement

{'level_1': 0.994,
 'level_2': 0.848,
 'level_3': 0.724,
 'test_labels_missing_from_tree': 7}

## HumSet: can documents leak across splits, and is a missing label really a "no"?

HumSet's analysts worked in separate projects, and a project may never have used part of the framework. If so, a
missing label does not always mean the label does not apply, which would make every method's precision look worse.

In [5]:
train_rows = ds.load_humset_rows(root, "train")
test_rows = ds.load_humset_rows(root, "test")
shared = set(test_rows.lead_id) & set(train_rows.lead_id)
if shared:
    raise ValueError(f"{len(shared)} test documents also appear in training")
projects = {}
for row in pd.concat([train_rows, test_rows]).to_dict("records"):
    projects.setdefault(row["project_id"], set()).update(ds.humset_labels(row))
leaf_count = len(humset.taxonomy.at_tier(2)) + sum(
    1 for root_id in humset.taxonomy.roots if root_id not in humset.taxonomy.children
)
usage = [len(labels) / leaf_count for labels in projects.values()]
results["humset"] = {
    "test_documents_in_train": len(shared),
    "projects": len(projects),
    "median_share_of_labels_a_project_ever_used": round(float(np.median(usage)), 3),
    "lowest_share": round(float(np.min(usage)), 3),
    "projects_using_every_label": int(sum(share == 1 for share in usage)),
    "entries_with_no_label": round(float(np.mean([not item.labels for item in humset.test])), 3),
}
results["humset"]

{'test_documents_in_train': 0,
 'projects': 46,
 'median_share_of_labels_a_project_ever_used': 0.548,
 'lowest_share': 0.226,
 'projects_using_every_label': 0,
 'entries_with_no_label': 0.046}

In [6]:
ex.save_results("01_test_sets", results);